In [36]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectPercentile, chi2

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report, accuracy_score, f1_score

In [20]:
data = pd.read_excel("../data/job_classification.ods", engine = "odf", dtype = str)

data.head()

,title,location,description,function,industry,career_level
0,Technical Professional Lead - Process,"Houston, TX","Responsible for the study, design, and specifi...",production_manufacturing,Machinery and Industrial Facilities Engineering,senior_specialist_or_project_manager
1,Cnslt - Systems Eng- Midrange 1,"Seattle, WA","Participates in design, development and implem...",information_technology_telecommunications,Financial Services,senior_specialist_or_project_manager
2,SharePoint Developers and Solution Architects,"Dallas, TX",We are currently in need of Developers who can...,consulting,IT Consulting,senior_specialist_or_project_manager
3,Business Information Services - Strategic Acco...,North Carolina,Experian is seeking an experienced Account Exe...,sales,"Security, Risk, Restructuring Consulting",senior_specialist_or_project_manager
4,Strategic Development Director (procurement),"Austin, TX",Â Want to join a world-class global procuremen...,procurement_materials_logistics,Information Technology,bereichsleiter


In [21]:
data = data.dropna(axis = 0)
data.shape

(8073, 6)

In [22]:
def filler_location(location):
    result = re.findall("\\,\\s[A-Z]{2}$", location)
    if len(result) > 0:
        return result[0][2:]
    else:
        return location


data["location"] = data["location"].apply(filler_location)

In [23]:
target = "career_level"
x = data.drop(target, axis = 1)
y = data[target]

In [24]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42, stratify = y)

In [25]:
transformers = ColumnTransformer(transformers = [
    ("title", TfidfVectorizer(stop_words="english"), "title"),
    ("location", OneHotEncoder(handle_unknown="ignore"), ["location"]),
    ("description", TfidfVectorizer(stop_words="english", ngram_range=(1,2), min_df = 0.01, max_df = 0.95), "description"),
    ("function", OneHotEncoder(handle_unknown="ignore"), ["function"]),
    ("industry", TfidfVectorizer(stop_words="english"), "industry")
])

In [51]:
logistic_model = Pipeline(steps = [
    ("transformers", transformers),
    ("feature_selector", SelectPercentile(chi2, percentile = 5)),
    ("classifier", LogisticRegression(max_iter = 1000))
])

In [52]:
logistic_model.fit(x_train, y_train)
y_logistic_predict = logistic_model.predict(x_test)
print(classification_report(y_test, y_logistic_predict))

                                        precision    recall  f1-score   support

                        bereichsleiter       0.55      0.38      0.45       192
         director_business_unit_leader       0.71      0.36      0.48        14
                   manager_team_leader       0.67      0.69      0.68       534
managing_director_small_medium_company       0.00      0.00      0.00         1
  senior_specialist_or_project_manager       0.84      0.91      0.87       868
                            specialist       0.00      0.00      0.00         6

                              accuracy                           0.76      1615
                             macro avg       0.46      0.39      0.41      1615
                          weighted avg       0.75      0.76      0.75      1615



C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control 

In [53]:
svc_model = Pipeline(steps=[
    ("transformers", transformers),
    ("feature_selector", SelectPercentile(chi2, percentile = 5)),
    ("classifier", LinearSVC())
])

In [54]:
svc_model.fit(x_train, y_train)
y_svc_predict = svc_model.predict(x_test)
print(classification_report(y_test, y_svc_predict))

                                        precision    recall  f1-score   support

                        bereichsleiter       0.53      0.36      0.43       192
         director_business_unit_leader       0.90      0.64      0.75        14
                   manager_team_leader       0.67      0.67      0.67       534
managing_director_small_medium_company       1.00      1.00      1.00         1
  senior_specialist_or_project_manager       0.84      0.91      0.87       868
                            specialist       1.00      0.17      0.29         6

                              accuracy                           0.76      1615
                             macro avg       0.82      0.62      0.67      1615
                          weighted avg       0.75      0.76      0.75      1615



In [55]:
rf_model = Pipeline(steps=[
    ("transformers", transformers),
    ("feature_selector", SelectPercentile(chi2, percentile = 5)),
    ("classifier", RandomForestClassifier(random_state=42))
])

In [56]:
rf_model.fit(x_train, y_train)
y_rf_predict = rf_model.predict(x_test)
print(classification_report(y_test, y_rf_predict))

                                        precision    recall  f1-score   support

                        bereichsleiter       0.61      0.12      0.20       192
         director_business_unit_leader       1.00      0.29      0.44        14
                   manager_team_leader       0.64      0.76      0.69       534
managing_director_small_medium_company       0.00      0.00      0.00         1
  senior_specialist_or_project_manager       0.84      0.91      0.88       868
                            specialist       1.00      0.17      0.29         6

                              accuracy                           0.76      1615
                             macro avg       0.68      0.37      0.42      1615
                          weighted avg       0.75      0.76      0.73      1615



C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control 

In [57]:
results = pd.DataFrame({
    "Experiment": [
        "Logistic Regression",
        "SVC",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_logistic_predict),
        accuracy_score(y_test, y_svc_predict),
        accuracy_score(y_test, y_rf_predict)
    ],
    "Macro F1": [
        f1_score(y_test, y_logistic_predict, average="macro"),
        f1_score(y_test, y_svc_predict, average="macro"),
        f1_score(y_test, y_rf_predict, average="macro")
    ],
    "Weighted F1": [
        f1_score(y_test, y_logistic_predict, average="weighted"),
        f1_score(y_test, y_svc_predict, average="weighted"),
        f1_score(y_test, y_rf_predict, average="weighted")
    ]
})

results

,Experiment,Accuracy,Macro F1,Weighted F1
0,Logistic Regression,0.761610,0.412477,0.751425
1,SVC,0.759133,0.668383,0.749494
2,Random Forest,0.759752,0.417193,0.730451
